In [0]:
spark.sql("CREATE TABLE IF NOT EXISTS workspace.default.raw_data")

DataFrame[]

In [0]:
# Cell 1 — pull the source files down
import urllib.request

year_month = "2026-05"
table_path = "/Volumes/workspace/default/raw_data"
trip_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year_month}.parquet"
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

urllib.request.urlretrieve(trip_url, f"{table_path}/yellow_tripdata_{year_month}.parquet")
urllib.request.urlretrieve(zone_url, f"{table_path}/taxi_zone_lookup.csv")
print("done")

done


In [0]:
# Cell 2 — look before you load
df_trips_raw = spark.read.parquet(f"{table_path}/yellow_tripdata_{year_month}.parquet")
df_trips_raw.printSchema()
df_trips_raw.display(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2,2026-05-01T00:04:59.000,2026-05-01T00:32:48.000,1,7.51,1,N,138,37,2,35.9,6.0,0.5,0.0,0.0,1.0,45.4,0.0,2.0,0.0
2,2026-05-01T00:37:05.000,2026-05-01T00:59:47.000,1,6.14,1,N,138,237,1,27.5,6.0,0.5,9.56,0.0,1.0,49.81,2.5,2.0,0.75
1,2026-05-01T00:34:05.000,2026-05-01T00:54:02.000,1,2.4,1,N,249,232,1,19.1,4.25,0.5,3.73,0.0,1.0,28.58,2.5,0.0,0.75
1,2026-05-01T00:55:07.000,2026-05-01T01:02:43.000,0,1.2,1,N,232,114,1,9.3,4.25,0.5,3.0,0.0,1.0,18.05,2.5,0.0,0.75
7,2026-05-01T00:44:13.000,2026-05-01T00:44:13.000,2,0.86,1,N,140,237,1,7.2,0.0,0.5,2.44,0.0,1.0,14.64,2.5,0.0,0.0
2,2026-05-01T00:24:35.000,2026-05-01T00:49:55.000,1,5.89,1,N,255,236,1,28.9,1.0,0.5,6.78,0.0,1.0,40.68,2.5,0.0,0.0
1,2026-05-01T00:06:08.000,2026-05-01T00:12:47.000,1,1.6,1,N,43,239,1,9.3,3.5,0.5,2.85,0.0,1.0,17.15,2.5,0.0,0.0
2,2026-05-01T00:49:20.000,2026-05-01T01:06:36.000,1,3.84,1,N,164,88,1,19.8,1.0,0.5,1.0,0.0,1.0,26.55,2.5,0.0,0.75
2,2026-05-01T00:29:38.000,2026-05-01T00:32:49.000,1,0.61,1,N,234,164,1,5.8,1.0,0.5,3.46,0.0,1.0,15.01,2.5,0.0,0.75
2,2026-05-01T00:39:07.000,2026-05-01T00:48:32.000,1,1.98,1,N,234,144,1,11.4,1.0,0.5,4.29,0.0,1.0,21.44,2.5,0.0,0.75


In [0]:
# Cell 3 — land as Bronze Delta tables
from pyspark.sql.functions import current_timestamp, lit

df_trips_bronze = (
    df_trips_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit(f"yellow_tripdata_{year_month}.parquet"))
)
# append mode — future months get added, not overwritten
df_trips_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_yellow_taxi")

df_zones = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{table_path}/taxi_zone_lookup.csv")
)
df_zones.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_taxi_zone_lookup")

print(f"Trips: {df_trips_bronze.count()}, Zones: {df_zones.count()}")

Trips: 4090836, Zones: 265


In [0]:
taxi_zone_lookup_df = (spark.read
                       .format("CSV")
                       .option("header", True)
                       .option("inferschema", True)
                       .load("/Volumes/workspace/default/raw_data/taxi_zone_lookup.csv")
                       )
taxi_zone_lookup_df.limit(10).display()

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
yellow_trip_df = (spark.read
                  .format("parquet")
                  .option("header", True)
                  .option("inferschema", True)
                  .load("/Volumes/workspace/default/raw_data/yellow_tripdata_2026-05.parquet")
                  )
yellow_trip_df.limit(10).display()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2,2026-05-01T00:04:59.000,2026-05-01T00:32:48.000,1,7.51,1,N,138,37,2,35.9,6.0,0.5,0.0,0.0,1.0,45.4,0.0,2.0,0.0
2,2026-05-01T00:37:05.000,2026-05-01T00:59:47.000,1,6.14,1,N,138,237,1,27.5,6.0,0.5,9.56,0.0,1.0,49.81,2.5,2.0,0.75
1,2026-05-01T00:34:05.000,2026-05-01T00:54:02.000,1,2.4,1,N,249,232,1,19.1,4.25,0.5,3.73,0.0,1.0,28.58,2.5,0.0,0.75
1,2026-05-01T00:55:07.000,2026-05-01T01:02:43.000,0,1.2,1,N,232,114,1,9.3,4.25,0.5,3.0,0.0,1.0,18.05,2.5,0.0,0.75
7,2026-05-01T00:44:13.000,2026-05-01T00:44:13.000,2,0.86,1,N,140,237,1,7.2,0.0,0.5,2.44,0.0,1.0,14.64,2.5,0.0,0.0
2,2026-05-01T00:24:35.000,2026-05-01T00:49:55.000,1,5.89,1,N,255,236,1,28.9,1.0,0.5,6.78,0.0,1.0,40.68,2.5,0.0,0.0
1,2026-05-01T00:06:08.000,2026-05-01T00:12:47.000,1,1.6,1,N,43,239,1,9.3,3.5,0.5,2.85,0.0,1.0,17.15,2.5,0.0,0.0
2,2026-05-01T00:49:20.000,2026-05-01T01:06:36.000,1,3.84,1,N,164,88,1,19.8,1.0,0.5,1.0,0.0,1.0,26.55,2.5,0.0,0.75
2,2026-05-01T00:29:38.000,2026-05-01T00:32:49.000,1,0.61,1,N,234,164,1,5.8,1.0,0.5,3.46,0.0,1.0,15.01,2.5,0.0,0.75
2,2026-05-01T00:39:07.000,2026-05-01T00:48:32.000,1,1.98,1,N,234,144,1,11.4,1.0,0.5,4.29,0.0,1.0,21.44,2.5,0.0,0.75
